### Wine Quality Dataset Analysis and Feature Engineering

#### 1. Clear Overview

The Wine Quality dataset is a real-world benchmark containing physicochemical measurements from wine samples in Portugal. It serves as a practical environment for practicing feature engineering and building classification models to predict human-assigned quality ratings on a scale of 3 to 8.

#### 2. Structured Table of Contents

- Data Loading and Preprocessing
- Correlation Analysis (Heatmaps)
- Feature Relationship Visualization (Box Plots)
- Model Training and Feature Importance

#### 3. Create Sections for Each Main Component

**Data Loading and Preprocessing**

The workflow begins by loading the dataset into a pandas DataFrame to assess data integrity.

- **Core Libraries:** `pandas`, `seaborn`, `matplotlib`.
- **Preprocessing:** The dataset is checked for missing values; in this specific instance, no features require imputation.

In [ ]:
# ---------------------------------------------------------
# STEP 1: IMPORT CORE LIBRARIES & LOAD DATA
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request

# Fetch the standard Wine Quality (Red) dataset from the UCI Repository 
# to ensure the code is instantly reproducible.
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
urllib.request.urlretrieve(url, 'wine_quality.csv')

# Load dataset (Note: the UCI dataset uses semicolons as separators)
df_wine = pd.read_csv('wine_quality.csv', sep=';')

# Verify no missing values exist in the dataset
missing_values = df_wine.isnull().sum().sum()
print(f"Total missing values in dataset: {missing_values}\n")

# Initial inspection: Display the first 5 rows
display(df_wine.head())

**Correlation Analysis (Heatmaps)**

Correlation heatmaps quantify the linear relationship between variables, helping identify features with the strongest predictive potential for the `quality` target.

- **Interpretation:** 
  - **Correlation Coefficient ($r$):** Ranges from $-1$ (perfect negative correlation) to $+1$ (perfect positive correlation). A value of $0$ indicates no linear correlation.
  - **Key Findings:** `Alcohol` and `Sulfates` exhibit strong positive correlations with wine quality ($r \approx 0.5 - 0.6$), while `Volatile Acidity` shows a significant negative correlation ($r \approx -0.4$ to $-0.6$).
  - **Feature Filtering:** Features like `Residual Sugar` and `Free Sulfur Dioxide` demonstrate low correlation with the target, marking them as candidates for removal to reduce model dimensionality.

In [ ]:
# ---------------------------------------------------------
# STEP 2: CORRELATION ANALYSIS (HEATMAP)
# ---------------------------------------------------------
# Compute the Pearson correlation matrix for all numerical features
corr_matrix = df_wine.corr()

# Set up the matplotlib figure size
plt.figure(figsize=(12, 10))

# Generate a custom diverging colormap to highlight positive/negative correlations
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    corr_matrix, 
    annot=True,        # Show the actual correlation values
    fmt=".2f",         # Format to 2 decimal places
    cmap=cmap,         # Apply the diverging palette
    vmax=1, vmin=-1,   # Anchor the colorbar limits to valid correlation bounds
    center=0,          # Center the colormap at zero
    square=True,       # Force cells to be square
    linewidths=.5      # Add slight spacing between cells for readability
)

plt.title('Feature Correlation Heatmap: Physicochemical Properties vs. Quality', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

**Feature Relationship Visualization (Box Plots)**

Box plots allow us to observe the distribution of a continuous feature across different discrete quality ratings.

- **Analytical Goal:** By plotting features like `Alcohol` or `Volatile Acidity` against quality scores (3 to 8), we can visually identify trends, variance, and outliers.
- **Utility:** This confirms whether a feature's behavior is consistent across quality levels, reinforcing the findings from the correlation heatmap.

In [ ]:
# ---------------------------------------------------------
# STEP 3: FEATURE RELATIONSHIP VISUALIZATION
# ---------------------------------------------------------
sns.set_theme(style="whitegrid")

# Create a 1x2 grid of subplots for side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Alcohol vs Quality (Positive Correlation)
sns.boxplot(
    data=df_wine, 
    x='quality', 
    y='alcohol', 
    ax=axes[0], 
    palette="Blues"
)
axes[0].set_title('Alcohol Content Across Quality Ratings (Positive Trend)', fontsize=14)
axes[0].set_xlabel('Quality Score (3-8)', fontsize=12)
axes[0].set_ylabel('Alcohol (% vol)', fontsize=12)

# Plot 2: Volatile Acidity vs Quality (Negative Correlation)
sns.boxplot(
    data=df_wine, 
    x='quality', 
    y='volatile acidity', 
    ax=axes[1], 
    palette="Reds"
)
axes[1].set_title('Volatile Acidity Across Quality Ratings (Negative Trend)', fontsize=14)
axes[1].set_xlabel('Quality Score (3-8)', fontsize=12)
axes[1].set_ylabel('Volatile Acidity (g/dm³)', fontsize=12)

plt.tight_layout()
plt.show()

**Model Training and Feature Importance**

We utilize a Random Forest Classifier to identify which chemical properties most significantly influence the predicted wine quality.

- **Standardization:** Before training, features are scaled using `StandardScaler`.
  - _Note:_ While tree-based models like Random Forest are theoretically invariant to feature scaling, standardizing is a common best practice in many pipelines to ensure compatibility with other estimators.
- **Feature Importance:** By extracting the `feature_importances_` attribute, we rank the variables.
  - **Top Predictors:** `Alcohol` ($\approx 0.15$) and `Sulfates` ($\approx 0.12$) emerge as the most critical features for the model’s predictive power.
  - **Dimensionality Reduction:** Features with negligible importance—such as `Free Sulfur Dioxide` and `Citric Acid`—can be safely removed to simplify the model and reduce training time.

In [ ]:
# ---------------------------------------------------------
# STEP 4: PREPROCESSING, MODELING, AND FEATURE IMPORTANCE
# ---------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target label (y)
X = df_wine.drop('quality', axis=1)
y = df_wine['quality']

# Initialize the standard scaler and fit_transform the feature set
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_scaled, y)

# Extract and sort the feature importances
feature_importances = rf_model.feature_importances_
df_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# ---------------------------------------------------------
# STEP 5: VISUALIZE FEATURE IMPORTANCES
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_importance, 
    x='Importance', 
    y='Feature', 
    palette='magma'
)

plt.title('Random Forest Derived Feature Importances (Wine Quality)', fontsize=16, pad=15)
plt.xlabel('Relative Importance Score', fontsize=14)
plt.ylabel('Physicochemical Feature', fontsize=14)
plt.tight_layout()
plt.show()

# Display the exact calculated values
display(df_importance)

#### 4. Application Summary

- **Dimensionality Reduction:** The analysis demonstrates that predictive accuracy is driven by a subset of features. Removing low-importance variables (e.g., Free Sulfur Dioxide, Residual Sugar) reduces noise and computational complexity.
- **Predictive Strength:** `Alcohol` and `Volatile Acidity` (followed closely by `Sulfates`) are consistently highlighted as primary drivers of wine quality across both correlation analysis and algorithmic importance scores.
- **Workflow:** This module illustrates the full end-to-end pipeline: exploration $\rightarrow$ visualization $\rightarrow$ preprocessing $\rightarrow$ model training $\rightarrow$ feature selection.